[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day2_Hydroeconomics_and_Water_Mapping/03_vegetation_and_water_spectral_indices_es.ipynb)

# Día 2 - Hidroeconomía y Mapeo del Agua
## Sesión 3: Índices espectrales de vegetación y agua

| | |
|---|---|
| **Fecha** | 2026-10-13 (martes) |
| **Horario** | 14:00 - 15:30 |
| **Entidad** | SEI / IMDEA |
| **Instructores** | Christhian, Sebastian |

### Contenido

> Índices espectrales de vegetación y agua (NDVI, EVI, SAVI, NDWI, NDMI, NDTI, NDCI): qué mide cada uno y cuándo usarlo. Series temporales de índices para fenología de cultivos, estrés hídrico y calidad del agua

### Objetivos de aprendizaje

- [ ] Escribir la ecuación de cada índice e identificar qué bandas espectrales usa
- [ ] Calcular los siete índices en una imagen Sentinel-2 y comparar sus patrones espaciales
- [ ] Aplicar el mismo cálculo a una colección de imágenes de un año con una sola función
- [ ] Extraer la serie temporal de un índice para una región dibujada e interpretarla en términos de fenología del cultivo y disponibilidad de agua

### Datos necesarios

No se necesitan archivos locales: las imágenes se leen del catálogo de Earth Engine
(`COPERNICUS/S2_SR_HARMONIZED`, reflectancia de superficie Sentinel-2 Nivel 2A).
Se requiere la cuenta de Earth Engine configurada en la sesión 2.

---


## 3.1 - Earth Engine y el área de estudio

Repetimos los tres primeros pasos de la sesión 2: conectarse a Earth Engine, abrir un mapa
interactivo y tomar el área de estudio de una geometría dibujada en ese mapa. Solo cambia el sensor -
aquí trabajamos exclusivamente con Sentinel-2, porque su píxel de 10-20 m y su revisita de 5 días son
adecuados para el monitoreo a escala de parcela.


In [ ]:
# ---------------------------------------------------------------------------
# Librerías (descomente la línea de pip en un entorno nuevo, p. ej. Google Colab)
# ---------------------------------------------------------------------------
#%pip install -q earthengine-api geemap

import ee
import geemap

print("ee", ee.__version__, "| geemap", geemap.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Autenticar e inicializar Earth Engine
# ---------------------------------------------------------------------------
# >>> REEMPLAZAR por el id de su propio proyecto de Cloud <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine connected - project:", EE_PROJECT)

### Seleccione el área de estudio

Use la herramienta de **marcador** o de **polígono** de la barra de la izquierda para ubicar su área de estudio.
Los índices se calculan por píxel, así que un punto es suficiente para ubicar la escena; la ventana
de análisis se construye a su alrededor en la celda siguiente.

Si no se dibuja nada, el notebook usa el punto de la sesión 2, en el altiplano
boliviano, para que se pueda ejecutar de principio a fin.


In [ ]:
# ---------------------------------------------------------------------------
# Mapa interactivo - dibuje aquí su área de estudio
# ---------------------------------------------------------------------------
DEFAULT_LON, DEFAULT_LAT = -68.7849, -16.5523   # el mismo punto usado en la sesión 2

Map = geemap.Map(center=[DEFAULT_LAT, DEFAULT_LON], zoom=12)
Map.add_basemap("SATELLITE")

Map

In [ ]:
# ---------------------------------------------------------------------------
# Área de estudio: la geometría dibujada, más una ventana de 5 km usada para recortar las visualizaciones
# ---------------------------------------------------------------------------
roi = Map.user_roi if Map.user_roi is not None else ee.Geometry.Point([DEFAULT_LON, DEFAULT_LAT])

CENTER_LON, CENTER_LAT = roi.centroid(maxError=1).coordinates().getInfo()
AOI = roi.buffer(5000).bounds()

Map.addLayer(AOI, {"color": "yellow"}, "AOI (5 km window)")
Map.centerObject(AOI, 12)

print("Centre  : lon = {:.4f}, lat = {:.4f}".format(CENTER_LON, CENTER_LAT))
print("AOI area: {:.1f} km2".format(AOI.area(maxError=1).divide(1e6).getInfo()))

## 3.2 - Colección Sentinel-2 y máscara de nubes

Aquí se fijan dos parámetros que se reutilizan en el resto del notebook:

- **Período**: un año, para que la serie temporal de la sección 3.6 cubra un ciclo de cultivo completo.
- **Cobertura máxima de nubes por escena**: 35%. Es un filtro *a nivel de escena* sobre el campo de metadatos
  `CLOUDY_PIXEL_PERCENTAGE`; descarta las imágenes que son mayormente nubes. No elimina
  las nubes que quedan dentro de las escenas aceptadas, que es la tarea de la máscara por píxel.

La máscara por píxel usa la banda **SCL** (Scene Classification Layer), producida por Sen2Cor,
que asigna una clase a cada píxel. Descartamos los píxeles saturados (1), la sombra de nubes (3),
las nubes de probabilidad media y alta (8, 9), los cirros finos (10) y la nieve/hielo (11), y conservamos
el resto. Los píxeles enmascarados no se ponen en cero: se eliminan del cálculo, de modo que una
media sobre una región los ignora en lugar de sesgarla hacia abajo.

La reflectancia se almacena como un entero y se reescala por `0.0001`. Los índices son cocientes, así que el
escalado se cancela en las formas de diferencia normalizada - pero no en el EVI, cuyos coeficientes
suponen reflectancia en el rango 0-1. Reescalar antes de calcular cualquier índice evita esta
inconsistencia.


In [ ]:
# ---------------------------------------------------------------------------
# Sentinel-2 Nivel 2A: filtrar, enmascarar nubes, reescalar a reflectancia
# ---------------------------------------------------------------------------
START = "2025-09-01"   # un año de datos
END   = "2026-09-01"
CLOUD_MAX = 35         # cobertura máxima de nubes por escena, en porcentaje

BANDS = ["B2", "B3", "B4", "B5", "B8", "B11", "B12"]   # azul, verde, rojo, red-edge 1, NIR, SWIR1, SWIR2

def mask_s2(image):
    # clases SCL a descartar: 1 saturado, 3 sombra, 8-9 nube, 10 cirros, 11 nieve
    scl = image.select("SCL")
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
                .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    optical = image.select(BANDS).multiply(0.0001)      # a reflectancia 0-1
    return ee.Image(optical.updateMask(clear)
                           .copyProperties(image, ["system:time_start",
                                                   "CLOUDY_PIXEL_PERCENTAGE"]))

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(roi)
        .filterDate(START, END)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
        .map(mask_s2))

print("Period          :", START, "to", END)
print("Cloud filter    : <", CLOUD_MAX, "%")
print("Images retained :", s2.size().getInfo())

In [ ]:
# ---------------------------------------------------------------------------
# Imagen de referencia: la escena con menos nubes del período
# ---------------------------------------------------------------------------
image = ee.Image(s2.sort("CLOUDY_PIXEL_PERCENTAGE").first()).clip(AOI)

print("Date        :", ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo())
print("Cloud cover :", round(image.get("CLOUDY_PIXEL_PERCENTAGE").getInfo(), 1), "%")

---

## 3.3 - Los índices

Un índice espectral es una combinación aritmética de bandas elegida para que la propiedad de
interés varíe más que los efectos que la perturban. La mayoría de los índices de esta sesión
tienen la forma de **diferencia normalizada**:

$$ND = \frac{\rho_A - \rho_B}{\rho_A + \rho_B}$$

donde $\rho$ es la reflectancia de superficie en las bandas $A$ y $B$. Esta forma está acotada entre
-1 y +1 y cancela los efectos *multiplicativos* que escalan ambas bandas por igual, como las
diferencias de iluminación y el sombreado topográfico. No cancela los efectos *aditivos*
como la radiancia atmosférica residual de trayecto, ni la contribución del fondo (suelo)
visto entre las plantas.

A continuación, cada índice se define con su ecuación, las bandas de Sentinel-2 que usa, el rango de
valores que se observa normalmente y su principal limitación.

---

### NDVI - Normalized Difference Vegetation Index (índice de vegetación de diferencia normalizada)

$$\text{NDVI} = \frac{\rho_{NIR} - \rho_{RED}}{\rho_{NIR} + \rho_{RED}}
= \frac{B8 - B4}{B8 + B4}$$

Usa el contraste entre la absorción de la clorofila en el rojo y la alta reflectancia del
mesófilo de la hoja en el infrarrojo cercano. Los valores menores que 0 se dan sobre el agua, alrededor de 0.1-0.2 sobre
suelo desnudo, y entre 0.5 y 0.9 sobre doseles cerrados.

*Limitaciones.* El NDVI se satura cuando el índice de área foliar supera aproximadamente 3: los
aumentos adicionales de biomasa producen poco cambio en el índice. En doseles dispersos, la reflectancia
del suelo entre las plantas contribuye a ambas bandas y desplaza el valor.

Rouse et al. (1973).

<img src="https://www.auravant.com/wp-content/uploads/2021/07/NDVI-values-by-crop-condition.jpeg
" width="500"> <img src="https://ece.montana.edu/seniordesign/archive/SP15/OpticalWeedMapping/uploads/4/9/2/7/49273335/1429843234.png
" width="300">

Fig 1- Índice NDVI Rouse et al. (1973).

---

### EVI - Enhanced Vegetation Index (índice de vegetación mejorado)

$$\text{EVI} = G \cdot \frac{\rho_{NIR} - \rho_{RED}}
{\rho_{NIR} + C_1 \rho_{RED} - C_2 \rho_{BLUE} + L}$$

con $G = 2.5$, $C_1 = 6$, $C_2 = 7.5$, $L = 1$, que para Sentinel-2 da:

$$\text{EVI} = 2.5 \cdot \frac{B8 - B4}{B8 + 6 \cdot B4 - 7.5 \cdot B2 + 1}$$

El término del azul corrige la dispersión por aerosoles, que afecta a la banda roja más que al infrarrojo
cercano; $L$ ajusta el efecto del fondo de suelo. El EVI responde a cambios en doseles densos
donde el NDVI ya se ha saturado.

*Limitaciones.* Depende de la banda azul, que tiene el mayor residuo atmosférico,
por lo que es más sensible a una corrección atmosférica imperfecta. Los coeficientes suponen
reflectancia en el rango 0-1.

Huete et al. (2002).

<img src="https://www.researchgate.net/publication/343758033/figure/fig4/AS:960336773533696@1605973537081/Seasonal-dynamics-of-the-Enhanced-Vegetation-Index-EVI-and-of-Ecosystem-Functional.png
" width="500">

Fig 2 - Ejemplo de EVI

---

### SAVI - Soil-Adjusted Vegetation Index (índice de vegetación ajustado al suelo)

$$\text{SAVI} = \frac{\rho_{NIR} - \rho_{RED}}{\rho_{NIR} + \rho_{RED} + L}
\cdot (1 + L), \qquad L = 0.5$$

Las mismas bandas que el NDVI con un factor de ajuste del suelo $L$ en el denominador. $L$ debería
acercarse a 0 para doseles densos y a 1 para coberturas muy dispersas; 0.5 es el compromiso habitual y
el valor usado aquí. El factor $(1+L)$ mantiene el rango comparable con el NDVI.

*Limitaciones.* $L$ se fija a priori y no se adapta a la densidad real del dosel, por lo que la
corrección es solo aproximada. Para coberturas dispersas y semiáridas el SAVI suele diferir del
NDVI más que para doseles cerrados.

Huete (1988).

---

### NDWI - Normalized Difference Water Index (McFeeters) (índice de agua de diferencia normalizada)

$$\text{NDWI} = \frac{\rho_{GREEN} - \rho_{NIR}}{\rho_{GREEN} + \rho_{NIR}}
= \frac{B3 - B8}{B3 + B8}$$

Diseñado para delimitar **cuerpos de agua abiertos**. El agua refleja moderadamente en el verde y
absorbe casi por completo en el infrarrojo cercano, por lo que da valores positivos, mientras que
la vegetación y el suelo dan valores negativos. Comúnmente se usa un umbral de 0 para separar el agua
de la tierra, aunque el valor óptimo depende de la escena.

*Limitaciones.* Las superficies construidas también producen valores positivos y se confunden con frecuencia
con el agua. Usar el SWIR en lugar del NIR (MNDWI, Xu 2006:
$(B3 - B11)/(B3 + B11)$) reduce esta confusión.

McFeeters (1996).

<style>
img[alt="image-2.png"] { width: 600px; }
</style>

![image-2.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/03_c11_image-2.png)

Fig 3 - Inundación en Porto Alegre, 2024, Brasil. [WebApp](https://ee-scriptsambgeo.projects.earthengine.app/view/floodpoaseries)


---

### NDMI - Normalized Difference Moisture Index (índice de humedad de diferencia normalizada)

$$\text{NDMI} = \frac{\rho_{NIR} - \rho_{SWIR1}}{\rho_{NIR} + \rho_{SWIR1}}
= \frac{B8 - B11}{B8 + B11}$$

Usa la absorción del agua líquida alrededor de 1.6 um: a medida que baja el contenido de agua de la hoja,
la reflectancia en el SWIR aumenta y el índice disminuye. Sigue el **contenido de agua del dosel** en lugar
de la cantidad de biomasa verde, y responde al estrés hídrico antes que el NDVI, porque
el dosel pierde agua antes de perder verdor.

*Limitaciones.* La señal mezcla el contenido de agua de la hoja con la estructura del dosel y, en coberturas
dispersas, con la humedad del suelo. Debe interpretarse como un indicador relativo en el tiempo para
el mismo objetivo, no como una medición absoluta del contenido de agua.

Gao (1996).


> **Una nota sobre los nombres.** El índice propuesto por Gao (1996) también se llamó NDWI, y tiene la
> misma fórmula que el NDMI de arriba. Por lo tanto, en la literatura circulan dos índices diferentes
> con el nombre NDWI: el de McFeeters (verde-NIR, para agua abierta) y el de Gao (NIR-SWIR, para el
> contenido de agua de la vegetación). Verifique siempre qué bandas se están usando.

---

### NDTI - Normalized Difference Turbidity Index (índice de turbidez de diferencia normalizada)

$$\text{NDTI} = \frac{\rho_{RED} - \rho_{GREEN}}{\rho_{RED} + \rho_{GREEN}}
= \frac{B4 - B3}{B4 + B3}$$

El sedimento en suspensión aumenta la reflectancia del agua en todo el rango visible, pero
aumenta más el rojo que el verde: el agua clara absorbe fuertemente en el rojo, mientras que las partículas
minerales dispersan en esa región. Por lo tanto, el índice aumenta con la **turbidez** y con la
concentración de sólidos suspendidos totales. El agua clara da valores negativos y el agua turbia
valores positivos; la relación con una concentración medida en mg/l es empírica y tiene que
calibrarse localmente.

*Limitaciones.* Solo tiene sentido sobre píxeles que ya se sabe que son agua, por lo que
debe aplicarse dentro de una máscara de agua - primero NDWI o MNDWI, después NDTI. El brillo solar especular (sun glint),
los efectos de adyacencia cerca de la orilla, la reflectancia del fondo en aguas someras y las floraciones densas de algas
aumentan la señal en el rojo y se leen como turbidez.

Lacaux et al. (2007).

![image.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/03_c11_image.png)

Fig 3 - Río Guaíba, Brasil, NDTI después de la lluvia.

---

### NDCI - Normalized Difference Chlorophyll Index (índice de clorofila de diferencia normalizada)

$$\text{NDCI} = \frac{\rho_{RE1} - \rho_{RED}}{\rho_{RE1} + \rho_{RED}}
= \frac{B5 - B4}{B5 + B4}$$

Construido para la **clorofila-a en aguas continentales y costeras**. El fitoplancton absorbe en el rojo
alrededor de 665 nm y produce un pico de reflectancia cerca de 700 nm; B5 (705 nm, la primera banda
red-edge) se ubica sobre ese pico mientras que B4 (665 nm) se ubica en el valle de absorción, de modo que la diferencia
entre ambas crece con la concentración de clorofila. Es uno de los pocos índices que
Sentinel-2 puede calcular y Landsat no, porque Landsat no tiene banda red-edge.

*Limitaciones.* La misma condición previa que el NDTI: solo píxeles de agua. Una turbidez fuerte aumenta ambas
bandas y debilita la señal, por eso el NDCI y el NDTI se leen juntos. La conversión a
mg/m3 de clorofila-a es empírica y específica del sitio, y el píxel de 20 m de B5 restringe el
índice a cuerpos de agua de al menos unos cientos de metros de ancho.

Mishra & Mishra (2012).

<style>
img[alt="image-3.png"] { width: 400px; }
</style>

![image-3.png](https://raw.githubusercontent.com/ScriptsRemote/Transcend_SummerSchool/main/Figures/Day2/03_c11_image-3.png)

Fig 4 - NDVI en el Lago Titicaca - Acceda al [código](https://code.earthengine.google.com/60bc96d560c6d837fa6871bb96943d49)

---

### Resumen

| Índice | Bandas (S2) | Mide | Rango típico |
|---|---|---|---|
| NDVI | B8, B4 | biomasa verde / vigor | -0.1 a 0.9 |
| EVI | B8, B4, B2 | biomasa verde, doseles densos | 0 a 1 |
| SAVI | B8, B4 | biomasa con corrección del suelo | -0.1 a 0.9 |
| NDWI | B3, B8 | agua abierta | -1 a 1, agua > 0 |
| NDMI | B8, B11 | contenido de agua del dosel | -0.5 a 0.5 |
| NDTI | B4, B3 | turbidez / sólidos suspendidos | -0.4 a 0.4, solo agua |
| NDCI | B5, B4 | clorofila-a en el agua | -0.2 a 0.5, solo agua |


---

## 3.4 - Un índice a la vez en la imagen de referencia

Calculamos los índices uno por uno en la imagen con menos nubes y agregamos cada resultado al mismo
mapa, `Map_idx`. El mapa se vuelve a mostrar al final de cada celda, de modo que la nueva capa
aparece inmediatamente debajo del código que la produjo; use el control de capas para cambiar
entre ellas.

Los tres índices de vegetación comparten la misma escala de colores, que es lo que los hace
comparables: cualquier diferencia en pantalla viene del índice, no de la visualización.

El NDTI y el NDCI van al final, y dentro de una máscara de agua construida a partir del NDWI: fuera del agua sus
valores están definidos aritméticamente pero no tienen sentido físico, y mostrarlos sobre
tierra lleva justamente a la mala interpretación que esta sesión intenta evitar.


In [ ]:
# ---------------------------------------------------------------------------
# NDVI = (B8 - B4) / (B8 + B4)
# ---------------------------------------------------------------------------
# normalizedDifference() implementa (primera - segunda) / (primera + segunda).
ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")

# Escala compartida para los tres índices de vegetación
vis_veg = {"min": -0.2, "max": 0.8,
           "palette": ["#f2efe4", "#cfe0ac", "#8fbc6a", "#3f8d47", "#12482a"]}

Map_idx = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=13)
Map_idx.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "True colour")
Map_idx.addLayer(ndvi, vis_veg, "NDVI")

Map_idx

In [ ]:
# ---------------------------------------------------------------------------
# EVI = 2.5 * (B8 - B4) / (B8 + 6*B4 - 7.5*B2 + 1)
# ---------------------------------------------------------------------------
# expression() se usa cuando la fórmula no es una simple diferencia normalizada.
evi = image.expression(
    "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
    {"NIR": image.select("B8"), "RED": image.select("B4"), "BLUE": image.select("B2")}
).rename("EVI")

Map_idx.addLayer(evi, vis_veg, "EVI")

# Compare con el NDVI sobre la vegetación más densa: el EVI sigue variando donde el NDVI se aplana.
Map_idx

In [ ]:
# ---------------------------------------------------------------------------
# SAVI = 1.5 * (B8 - B4) / (B8 + B4 + 0.5)      [ (1+L) * ... con L = 0.5 ]
# ---------------------------------------------------------------------------
savi = image.expression(
    "1.5 * (NIR - RED) / (NIR + RED + 0.5)",
    {"NIR": image.select("B8"), "RED": image.select("B4")}
).rename("SAVI")

Map_idx.addLayer(savi, vis_veg, "SAVI")

# La diferencia con el NDVI es mayor sobre coberturas dispersas, donde el suelo contribuye más.
Map_idx

In [ ]:
# ---------------------------------------------------------------------------
# NDWI = (B3 - B8) / (B3 + B8)
# ---------------------------------------------------------------------------
ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")

vis_water = {"min": -0.5, "max": 0.5,
             "palette": ["#eae4d6", "#b9d3e4", "#5f9fcb", "#1f6aa5", "#08355e"]}

Map_idx.addLayer(ndwi, vis_water, "NDWI")

# Agregar la máscara de agua obtenida con el umbral habitual, para inspeccionarla de forma crítica
Map_idx.addLayer(ndwi.updateMask(ndwi.gt(0)), {"palette": ["#1f6aa5"]}, "NDWI > 0 (water)")

Map_idx

In [ ]:
# ---------------------------------------------------------------------------
# NDMI = (B8 - B11) / (B8 + B11)
# ---------------------------------------------------------------------------
ndmi = image.normalizedDifference(["B8", "B11"]).rename("NDMI")

# Escala divergente: marrón = seco, neutro = intermedio, azul = alto contenido de agua
vis_ndmi = {"min": -0.5, "max": 0.5,
            "palette": ["#8c5a2b", "#c9a97a", "#efeeec", "#7fb0d0", "#14507d"]}

Map_idx.addLayer(ndmi, vis_ndmi, "NDMI")

Map_idx

In [ ]:
# ---------------------------------------------------------------------------
# NDTI = (B4 - B3) / (B4 + B3)        [solo píxeles de agua]
# ---------------------------------------------------------------------------
# La turbidez solo está definida sobre el agua, así que reutilizamos el umbral de NDWI de la
# celda anterior como máscara y calculamos el índice dentro de ella.
water_mask = ndwi.gt(0)

ndti = image.normalizedDifference(["B4", "B3"]).rename("NDTI").updateMask(water_mask)

# Escala secuencial: azul = agua clara, ocre = agua turbia
vis_ndti = {"min": -0.3, "max": 0.3,
            "palette": ["#0b3d62", "#3f7fa6", "#9dbfc9", "#d9c08a", "#a9782f"]}

Map_idx.addLayer(ndti, vis_ndti, "NDTI (water only)")

Map_idx

In [ ]:
# ---------------------------------------------------------------------------
# NDCI = (B5 - B4) / (B5 + B4)        [solo píxeles de agua]
# ---------------------------------------------------------------------------
# B5 es la primera banda red-edge (705 nm, 20 m), la que Landsat no tiene.
ndci = image.normalizedDifference(["B5", "B4"]).rename("NDCI").updateMask(water_mask)

# Escala secuencial: oscuro = oligotrófico, amarillo-verde = alta clorofila-a
vis_ndci = {"min": -0.1, "max": 0.4,
            "palette": ["#2b1a4a", "#2e6f8e", "#4aa06b", "#9ac544", "#f2e63d"]}

Map_idx.addLayer(ndci, vis_ndci, "NDCI (water only)")

# Lea el NDTI y el NDCI juntos: el sedimento y las algas aclaran el agua, pero
# solo la clorofila produce el pico en el red-edge que mide el NDCI.
Map_idx

### Los siete índices lado a lado

La misma imagen, la misma fecha y la misma región en cada panel: solo cambia la combinación
de bandas. El desplazamiento y el zoom están sincronizados, así que cualquier diferencia en pantalla viene
del índice.

Los tres paneles de vegetación comparten una escala de colores y se pueden comparar entre sí.
NDWI, NDMI, NDTI y NDCI usan sus propias escalas, así que **los colores no son comparables entre
esos paneles** ni con los de vegetación - compare los patrones espaciales, no los tonos.
El NDTI y el NDCI se dibujan solo dentro de la máscara de agua, así que todo lo que está fuera del agua es
transparente.

In [ ]:
# Misma imagen, misma fecha, misma escala de colores: solo cambia la fórmula.
vegetation = geemap.linked_maps(
    rows=1, cols=3, height="400px",
    center=[CENTER_LAT, CENTER_LON], zoom=13,
    ee_objects=[ndvi, evi, savi],
    vis_params=[vis_veg, vis_veg, vis_veg],
    labels=["NDVI (B8, B4)", "EVI (B8, B4, B2)", "SAVI (B8, B4)"],
    label_position="topright",
)

vegetation

In [ ]:
# ---------------------------------------------------------------------------
# Índices de agua, humedad y calidad del agua lado a lado
# ---------------------------------------------------------------------------
# los atributos de límites (bounds) de ipyleaflet rechazan el null que envía el front-end mientras un
# mapa en cuadrícula todavía no tiene tamaño - se permite para que la sincronización no genere un error.
for _t in ("north", "south", "east", "west"):
    geemap.Map.class_traits()[_t].allow_none = True

# Cada panel tiene su propia escala de colores, así que compare los patrones espaciales
# entre ellos, no los tonos. El NDTI y el NDCI llevan la máscara de agua de las
# celdas anteriores, así que el mapa base se ve en todo lo que no es agua.
water = geemap.linked_maps(
    rows=2, cols=2, height="400px",
    center=[CENTER_LAT, CENTER_LON], zoom=13,
    ee_objects=[ndwi, ndmi, ndti, ndci],
    vis_params=[vis_water, vis_ndmi, vis_ndti, vis_ndci],
    labels=["NDWI (B3, B8) - open water", "NDMI (B8, B11) - canopy water content",
            "NDTI (B4, B3) - turbidity, water only",
            "NDCI (B5, B4) - chlorophyll-a, water only"],
    label_position="topright",
)

water

### Qué comparar

1. **NDVI, EVI y SAVI.** Sobre vegetación densa, el panel de NDVI tiende a aplanarse en un
   tono oscuro uniforme mientras que el EVI todavía separa gradaciones - es la saturación descrita
   en 3.3. Sobre coberturas dispersas, compare el NDVI y el SAVI: la diferencia ahí es la contribución
   del suelo que el término $L$ está corrigiendo.
2. **NDWI.** Verifique la extensión de los valores positivos. Además del agua abierta, busque superficies
   construidas que devuelvan valores positivos - la confusión que el MNDWI resuelve reemplazando el
   NIR por el SWIR.
3. **NDMI frente a NDVI.** Áreas con NDVI similar pueden diferir en el NDMI. Donde eso ocurre,
   los doseles tienen un verdor comparable pero un contenido de agua diferente, que es la
   situación que el NDVI por sí solo no puede distinguir.
4. **NDTI frente a NDCI.** Ambos se muestran solo dentro de la máscara de agua del NDWI. Un embalse que
   es brillante en el NDTI pero bajo en el NDCI transporta sedimento mineral; un NDCI alto con un NDTI bajo
   indica en cambio una floración de algas. Observe también la orilla - los píxeles mixtos tierra/agua
   a lo largo del borde son donde ambos índices son menos confiables.
5. **El panel en color verdadero.** Úselo para confirmar que lo que marca un índice corresponde a un
   elemento real en el terreno, y no a una sombra, un borde de nube o un píxel enmascarado.

---

## 3.5 - El mismo cálculo sobre toda la colección

Aplicar los índices a una imagen responde *dónde*. Para responder *cuándo* - cuándo emerge el cultivo,
cuándo alcanza la cobertura máxima, cuándo baja el contenido de agua - el mismo cálculo debe
repetirse para cada imagen de la colección.

Escribimos una función que recibe una imagen y devuelve la misma imagen con las siete bandas de índices
añadidas, y la pasamos a `map()`. Earth Engine la aplica a cada imagen en el
servidor; en este momento no se descarga nada.


In [ ]:
# ---------------------------------------------------------------------------
# Una función, aplicada a cada imagen de la colección
# ---------------------------------------------------------------------------
INDICES = ["NDVI", "EVI", "SAVI", "NDWI", "NDMI"]   # serie de la sección 3.6
QUALITY_INDICES = ["NDTI", "NDCI"]                  # serie de la sección 3.7

def add_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = image.normalizedDifference(["B3", "B8"]).rename("NDWI")
    ndmi = image.normalizedDifference(["B8", "B11"]).rename("NDMI")
    # No se necesita máscara para estos dos: en 3.7 se reducen sobre un
    # polígono dibujado dentro de un cuerpo de agua, así que la geometría es la máscara de agua.
    ndti = image.normalizedDifference(["B4", "B3"]).rename("NDTI")
    ndci = image.normalizedDifference(["B5", "B4"]).rename("NDCI")
    evi = image.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {"NIR": image.select("B8"), "RED": image.select("B4"), "BLUE": image.select("B2")}
    ).rename("EVI")
    savi = image.expression(
        "1.5 * (NIR - RED) / (NIR + RED + 0.5)",
        {"NIR": image.select("B8"), "RED": image.select("B4")}
    ).rename("SAVI")
    return image.addBands([ndvi, evi, savi, ndwi, ndmi, ndti, ndci])

s2_indices = s2.map(add_indices)

print("Images   :", s2_indices.size().getInfo())
print("Bands now:", s2_indices.first().bandNames().getInfo())

---

## 3.6 - Series temporales de índices para una región

### Paso 1 - Dibuje la región a monitorear

Dibuje un **polígono** sobre un solo objetivo: una parcela de riego, un embalse, un tramo de
vegetación ribereña. Manténgalo dentro de una superficie homogénea - un polígono que cubre una parcela, un
camino y parte de un río promedia los tres juntos y la serie resultante no es
interpretable.

Se muestra la capa de NDVI para ayudar a delimitar los bordes.


In [ ]:
# ---------------------------------------------------------------------------
# Dibuje aquí la región para la que se extraerá la serie temporal
# ---------------------------------------------------------------------------
Map_ts = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=14)
Map_ts.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "True colour")
Map_ts.addLayer(ndvi, vis_veg, "NDVI")

Map_ts

### Paso 2 - Reducir cada imagen a un valor por índice

`reduceRegion()` reduce todos los píxeles de una región a una sola estadística. Aplicado a
cada imagen de la colección, convierte una pila de imágenes en una tabla: una fila por fecha,
una columna por índice.

Dos puntos a tener en cuenta:

- `scale = 20` coincide con la banda más gruesa usada (B11, a 20 m). Definir una escala más fina que los
  datos no agrega información y aumenta el cálculo.
- Los píxeles enmascarados por nubes se excluyen de la media. Si toda una región está enmascarada en una
  fecha dada, el reductor no devuelve ningún valor y esa fecha se elimina de la tabla. Por eso
  la serie tiene menos fechas que imágenes tiene la colección.


In [ ]:
region = Map_ts.user_roi if Map_ts.user_roi is not None else AOI

In [ ]:
# ---------------------------------------------------------------------------
# De una colección de imágenes a una tabla: media de cada índice por fecha
# ---------------------------------------------------------------------------
import pandas as pd


def index_means(image):
    stats = image.select(INDICES).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region, scale=20, maxPixels=1e9)
    return ee.Feature(None, stats).set("date", image.date().format("YYYY-MM-dd"))

series = s2_indices.map(index_means)

df = pd.DataFrame([f["properties"] for f in series.getInfo()["features"]])
df["date"] = pd.to_datetime(df["date"])
df = df.dropna().sort_values("date").set_index("date")[INDICES]

print("Images in the collection :", s2_indices.size().getInfo())
print("Dates with valid data    :", len(df))
df.head()

### Paso 3 - Graficar la serie

Los índices se separan en dos paneles porque miden propiedades diferentes y no son
directamente comparables: los tres índices de vegetación comparten una escala y se pueden comparar
entre sí, mientras que el NDWI y el NDMI responden preguntas diferentes y tienen sus propios rangos.

La línea discontinua en 0 del panel inferior es el umbral que se usa comúnmente con el NDWI para separar
el agua de la tierra.


In [ ]:
# ---------------------------------------------------------------------------
# Series temporales de los índices sobre la región dibujada
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt

VEG   = {"NDVI": "#1baf7a", "EVI": "#eb6834", "SAVI": "#2a78d6"}
WATER = {"NDWI": "#4a3aa7", "NDMI": "#e34948"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#d9d8d2"

fig, (ax_v, ax_w) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                 gridspec_kw={"hspace": 0.28})

for ax, group, title in [(ax_v, VEG,   "Vegetation indices"),
                         (ax_w, WATER, "Water and moisture indices")]:
    for name, color in group.items():
        ax.plot(df.index, df[name], color=color, linewidth=2, marker="o",
                markersize=4, markeredgecolor="white", markeredgewidth=0.8,
                label=name, zorder=3)
    ax.set_title(title, fontsize=11, color=INK, loc="left", pad=6)
    ax.set_ylabel("Index value", fontsize=10, color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.legend(frameon=False, loc="upper left", fontsize=9, ncol=3, labelcolor=INK)

ax_w.axhline(0, color=MUTED, linewidth=1, linestyle="--", zorder=1)

fig.suptitle("Index time series over the drawn region", fontsize=13, color=INK,
             x=0.09, ha="left", y=0.97)
fig.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.08)
plt.show()

### Lectura de la serie

Trabaje estas preguntas con su propia región:

1. **Fenología.** ¿Muestra el NDVI un ciclo de crecimiento, máximo y senescencia? Ubique la fecha
   de cobertura máxima. Si la región es una parcela de riego con dos ciclos por año, deberían verse
   dos picos.
2. **NDVI, EVI y SAVI.** ¿Tienen las tres curvas la misma forma? Donde divergen, ¿es
   en el pico (saturación del NDVI) o al inicio y al final del ciclo (contribución del suelo)?
3. **NDMI antes que NDVI.** Durante un descenso, verifique si el NDMI empieza a bajar antes que el NDVI. Una
   caída del contenido de agua del dosel que precede a la pérdida de verdor es coherente con el
   inicio del estrés hídrico.
4. **Vacíos.** Cuente los intervalos sin datos. En la época de lluvias, la cobertura de nubes puede dejar vacíos
   de varias semanas - una limitación de la teledetección óptica que tiene que informarse junto con
   los resultados, no ocultarse.


---

## 3.7 - Series de calidad del agua sobre un cuerpo de agua

El NDTI y el NDCI se dejaron fuera de la serie anterior a propósito. Describen el estado de la
columna de agua, así que sobre una parcela o un tramo de matorral sus valores son aritméticos sin
sentido físico, y ponerlos en el mismo gráfico que el NDVI lleva justamente a la mala interpretación
que esta sesión intenta evitar.

En cambio, tienen su propia región. La lógica es la usada en 3.6 - dibujar, reducir, graficar -
con una sola diferencia: el polígono se dibuja **dentro de un cuerpo de agua**. Como cada píxel
de ese polígono es agua por construcción, **aquí no se necesita un umbral de NDWI**: la geometría
es la máscara.

### Paso 1 - Dibuje el cuerpo de agua

Dibuje un **polígono estrictamente dentro** de un embalse, un lago o un tramo ancho de río, manteniéndose a unos
píxeles de la orilla - los píxeles mixtos tierra/agua a lo largo de la orilla contaminan ambos
índices. La capa de NDWI se muestra para ayudarle a encontrar la orilla, no para enmascarar nada.

In [ ]:
# ---------------------------------------------------------------------------
# Dibuje aquí el cuerpo de agua para el que se extraerá la serie de calidad
# ---------------------------------------------------------------------------
# El NDWI se muestra solo como guía para ubicar la orilla. La máscara es el
# polígono que usted dibuja, así que manténgalo bien dentro del agua.
Map_water = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=14)
Map_water.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.3}, "True colour")
Map_water.addLayer(ndwi, vis_water, "NDWI")

Map_water

In [ ]:
# ---------------------------------------------------------------------------
# De la colección a una tabla: turbidez y clorofila-a medias por fecha
# ---------------------------------------------------------------------------
# El mismo reductor que en 3.6, aplicado a las dos bandas de calidad del agua sobre el
# polígono. No se aplica ninguna máscara: el polígono es la máscara de agua.
if Map_water.user_roi is not None:
    water_region = Map_water.user_roi
    print("Using the polygon you drew.")
else:
    water_region = AOI
    print("No polygon drawn -> falling back to the whole AOI.")
    print("The series below only means something over water: draw a polygon")
    print("inside a water body and run this cell again.")


def quality_means(image):
    stats = image.select(QUALITY_INDICES).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=water_region, scale=20, maxPixels=1e9)
    return ee.Feature(None, stats).set("date", image.date().format("YYYY-MM-dd"))

water_series = s2_indices.map(quality_means)

dfw = pd.DataFrame([f["properties"] for f in water_series.getInfo()["features"]])
dfw["date"] = pd.to_datetime(dfw["date"])
dfw = dfw.dropna().sort_values("date").set_index("date")[QUALITY_INDICES]

print("Dates with valid data :", len(dfw))
dfw.head()

In [ ]:
# ---------------------------------------------------------------------------
# Series de calidad del agua sobre el cuerpo de agua dibujado
# ---------------------------------------------------------------------------
# Un solo panel: ambos índices son diferencias normalizadas sobre los mismos píxeles
# de agua, así que comparten un eje y se pueden comparar entre sí.
QUALITY = {"NDTI": "#a9782f", "NDCI": "#2e8b7a"}

fig, ax = plt.subplots(figsize=(11, 4.2))

for name, color in QUALITY.items():
    ax.plot(dfw.index, dfw[name], color=color, linewidth=2, marker="o",
            markersize=4, markeredgecolor="white", markeredgewidth=0.8,
            label=name, zorder=3)

ax.set_ylabel("Index value", fontsize=10, color=MUTED)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, loc="upper left", fontsize=9, ncol=2, labelcolor=INK)
ax.set_title("Turbidity and chlorophyll-a over the drawn water body", fontsize=13,
             color=INK, loc="left", pad=10)

fig.subplots_adjust(left=0.09, right=0.98, top=0.86, bottom=0.14)
plt.show()

### Lectura de la serie de calidad del agua

1. **Sedimento.** Un aumento del NDTI poco después de las primeras lluvias es la carga de sedimento que llega
   con la escorrentía. Compare su momento con la serie de NDVI de la sección 3.6: el suelo desnudo en la
   cuenca al inicio de la temporada y un pico de turbidez en el embalse son los dos
   extremos del mismo proceso.
2. **Algas.** Un aumento del NDCI durante los meses cálidos, sin un aumento correspondiente del NDTI, indica
   una floración de fitoplancton y no la entrada de lodo. Cuando los dos aumentan juntos la señal
   es ambigua, y ninguno de los dos índices por sí solo puede separarlos.
3. **La orilla.** Repita la extracción con un polígono dibujado más cerca de la orilla. Si la
   serie cambia notablemente, los píxeles mixtos tierra/agua están determinando el resultado y el
   primer polígono es el confiable.
4. **Nivel del agua.** En un embalse que se reduce durante la época seca, un polígono fijo puede terminar
   parcialmente sobre el lecho seco. Revise la imagen en color verdadero de las últimas fechas antes de interpretar un
   aumento del NDTI como turbidez.
5. **Vacíos.** Como en 3.6, la cobertura de nubes elimina fechas de la serie. Sobre el agua el problema es
   mayor, porque la sombra de nubes sobre una superficie oscura es más difícil de enmascarar y lo que pasa el
   filtro no siempre está limpio.

---

## 3.8 - Ejercicio

**Consigna**

Elija **dos regiones contrastantes** dentro de su área de estudio - por ejemplo una parcela de riego
y una de secano, o un embalse y sus alrededores - y compare sus series de índices.

1. Dibuje la primera región, extraiga la serie y guárdela con otro nombre
   (por ejemplo `df_a = df.copy()`).
2. Dibuje la segunda región y repita, guardándola como `df_b`.
3. Grafique el mismo índice (NDVI, o NDMI) para las dos regiones en un solo gráfico.
4. Responda:
   - ¿Tienen las dos regiones el mismo ciclo fenológico? ¿Coinciden los picos en el tiempo?
   - ¿La diferencia entre ellas es constante a lo largo del año, o aparece solo en una
     estación específica?
   - ¿Qué índice separa las dos regiones con mayor claridad, y en qué época del año?

**Entregable:** el gráfico de comparación más un párrafo (10-15 líneas) que indique
lo que muestran las series y qué limitaciones afectan la interpretación (vacíos por nubes, tamaño del
polígono, píxeles mixtos).


---

## Referencias

**Índices**

- Rouse, J.W., Haas, R.H., Schell, J.A., Deering, D.W. (1974). Monitoring vegetation systems
  in the Great Plains with ERTS. *Third ERTS Symposium*, NASA SP-351, 309-317. (NDVI)
- Huete, A.R. (1988). A soil-adjusted vegetation index (SAVI). *Remote Sensing of
  Environment*, 25(3), 295-309.
- Huete, A., Didan, K., Miura, T., Rodriguez, E.P., Gao, X., Ferreira, L.G. (2002). Overview
  of the radiometric and biophysical performance of the MODIS vegetation indices. *Remote
  Sensing of Environment*, 83(1-2), 195-213. (EVI)
- McFeeters, S.K. (1996). The use of the Normalized Difference Water Index (NDWI) in the
  delineation of open water features. *International Journal of Remote Sensing*, 17(7),
  1425-1432.
- Gao, B.-C. (1996). NDWI - A normalized difference water index for remote sensing of
  vegetation liquid water from space. *Remote Sensing of Environment*, 58(3), 257-266. (NDMI)
- Xu, H. (2006). Modification of normalised difference water index (NDWI) to enhance open
  water features in remotely sensed imagery. *International Journal of Remote Sensing*,
  27(14), 3025-3033. (MNDWI)
- Lacaux, J.P., Tourre, Y.M., Vignolles, C., Ndione, J.A., Lafaye, M. (2007). Classification
  of ponds from high-spatial resolution remote sensing: application to Rift Valley Fever
  epidemics in Senegal. *Remote Sensing of Environment*, 106(1), 66-74. (NDTI)
- Mishra, S., Mishra, D.R. (2012). Normalized difference chlorophyll index: a novel model
  for remote estimation of chlorophyll-a concentration in turbid productive waters.
  *Remote Sensing of Environment*, 117, 394-406. (NDCI)

**Datos y herramientas**

- Sentinel-2 SR Harmonized: <https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED>
- Algoritmo Sentinel-2 Nivel 2A (clases SCL): <https://sentinels.copernicus.eu/web/sentinel/technical-guides/sentinel-2-msi/level-2a/algorithm>
- Guía de Earth Engine - reducción de una ImageCollection: <https://developers.google.com/earth-engine/guides/reducers_reduce_region>
- Documentación de geemap: <https://geemap.org>
